In [ ]:
!pip install torch datasets tokenizers

import torch
import torch.nn as nn
from datasets import load_dataset
import math

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=4, block_size=128):
        super().__init__()
        self.block_size = block_size
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(block_size, d_model)

        # We use standard layers, but we will pass a mask to force causal behavior
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        # 1. Embeddings + Positional Encoding
        x = self.embed(x) + self.pos(pos)

        # 2. The Causal Mask (CRITICAL for Next-Word Prediction)
        # Creates a matrix with -inf above the diagonal, preventing future sight.
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)

        # 3. Pass through transformer with the mask
        x = self.transformer(x, mask=causal_mask, is_causal=True)

        # 4. Final layer norm and projection to vocabulary
        x = self.ln(x)
        return self.fc(x)

# --- Data Preparation ---
print("Loading data...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
text = " ".join(dataset["train"]["text"])[:500000]  # Small subset for quick testing

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s): return [stoi[c] for c in s if c in stoi]
def decode(l): return "".join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Dataset length: {len(data)} characters. Vocab size: {vocab_size}")

# --- Training Setup ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 128
batch_size = 64 # Bumped up slightly for GPU efficiency

model = TinyGPT(vocab_size=vocab_size, d_model=256, n_heads=4, n_layers=4, block_size=block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# --- Training Loop ---
print("Starting training...")
model.train()
for step in range(2001):
    xb, yb = get_batch()

    logits = model(xb)
    loss = loss_fn(logits.view(-1, logits.size(-1)), yb.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Gradient clipping to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 250 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

# --- Generation ---
def generate(model, start="The meaning of life is", max_new=100):
    model.eval()
    x = torch.tensor(encode(start), dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_new):
            # Crop context to block_size to prevent out-of-bounds positional embeddings
            x_cond = x[:, -block_size:]

            logits = model(x_cond)
            # Focus only on the last time step
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat((x, next_token), dim=1)

    return decode(x[0].tolist())

print("\n--- Generation Test ---")
print(generate(model, start="The architecture of", max_new=200))

# --- Save and Quantize ---
print("\nSaving and Quantizing...")
torch.save(model.state_dict(), "tiny_gpt_raw.pth")

# PyTorch dynamic quantization requires the model to be on the CPU
model.to("cpu")
model.eval()
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)
torch.save(quantized_model.state_dict(), "tiny_gpt_quantized.pth")

import os
raw_size = os.path.getsize("tiny_gpt_raw.pth") / (1024 * 1024)
quant_size = os.path.getsize("tiny_gpt_quantized.pth") / (1024 * 1024)
print(f"Raw Model Size: {raw_size:.2f} MB")
print(f"Quantized Size: {quant_size:.2f} MB")

Loading data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset length: 500000 characters. Vocab size: 170
Starting training...
Step 0 | Loss: 5.1581
Step 250 | Loss: 2.3569
Step 500 | Loss: 1.9869
Step 750 | Loss: 1.8413
Step 1000 | Loss: 1.6760
Step 1250 | Loss: 1.6219
Step 1500 | Loss: 1.5423
Step 1750 | Loss: 1.5282
Step 2000 | Loss: 1.4786

--- Generation Test ---
The architecture of Housurya , pistone the south of goals recorded of figured in the Great ṯ books . Extrō to the carved , as revements building pregotive to be ecountitue . These were used by the sulpo general which th

Saving and Quantizing...
Raw Model Size: 12.53 MB
Quantized Size: 6.41 MB


/tmp/ipykernel_2132/3797197286.py:126: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [ ]:
# 1. Re-initialize the architecture with the same specs
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_raw = TinyGPT(vocab_size=vocab_size, d_model=256, n_heads=4, n_layers=4).to(device)

# 2. Load the weights
model_raw.load_state_dict(torch.load("tiny_gpt_raw.pth"))
model_raw.eval() # CRITICAL: Set to evaluation mode

# 3. Generate!
prompt = "The future of AI is"
generated_text = generate(model_raw, start=prompt, max_new=150)
print(generated_text)

The future of AI is there sciendures carder . " The March Time high a figure ( allush ) based by Shiva team for the steak on Way Unders . Madero themse of stated at the 


In [ ]:
prompt = "Who are you"
generated_text = generate(model_raw, start=prompt, max_new=150)
print(generated_text)

Who are young himself control , both weomengs the privation of Shiva . The ships were day Islands wented to subjeck of Microtic Balkyricisa . The Trujillo determ


In [ ]:
prompt = "Capital of France is?"
generated_text = generate(model_raw, start=prompt, max_new=150)
print(generated_text)

Capital of France is?ues country , forces the November 10 , and grow @-@ dollars of Women ; in Geehages ; War Way Lied Tettle Petford Creative Rock Fey being the gog it a 
